# EAGLE-3 vs TALON Speculative Decoding Benchmark

## 🚀 Quick Start for Batch Size 8 Test

**Run order:**
1. **Cells 1-4**: Setup (GPU check, Drive, code, environment)
2. **Cell 10**: Setup SGLang environment + register TALON plugin
3. **Cell 11**: **RUN THIS FIRST** - Batch size 8 test with EAGLE-3 vs TALON
4. **Cell 12**: View batch 8 results
5. **If Cell 11 succeeds**: Run Cell 13 for full sweep (batch 1/8/32)

---

## Background

This notebook benchmarks **TALON** (adaptive draft trees) vs **static EAGLE-3** speculative decoding.

**TALON Paper:** Liu et al., "TALON: Confidence-Aware Speculative Decoding with Adaptive Token Trees" (arXiv 2601.07353)

### Key Differences
| Method | Tree Shape | Selection Rule |
|--------|-----------|----------------|
| EAGLE-3 (static) | Fixed K=10, D=8, N=60 | Top-K at each layer |
| TALON-S (adaptive) | Dynamic, budget N=60 | Confidence-gated: keep nodes where p(u) >= mu*m_d |

### Algorithm 2 (TALON)
- **Layer 0 (robust init)**: Force top-K children of root (K=10)
- **Layer d >= 1 (confidence gating)**: Keep every child whose cumulative probability >= mu * m_d
- **Budget check**: At every layer, if budget exceeded, keep only top (N - |T|) by score

### Paths
- **Path A (SGLang)**: Batch sizes 1, 8, 32 - requires A100 (40GB) - uses TALON-S
- **Path B (EAGLE repo)**: Batch size 1 only - full TALON

### Before You Start
1. **A100 GPU** required for batch sizes > 1 (Colab Pro/Pro+)
2. Accept Llama 3.1 license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
3. Add `HF_TOKEN` in Colab Secrets (left sidebar 🔑)
4. Upload `specdec-bench.zip` to `MyDrive/specdec/`

In [ ]:
# ============================================================
# CELL 1: GPU CHECK
# ============================================================
# Must have A100 (40GB) for batch size 8 testing
# ============================================================

!nvidia-smi

import torch, subprocess, re
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 GPU"

name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
drv = re.search(r"CUDA Version: (\d+)\.(\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
CUDA_MAJOR = int(drv.group(1))

print(f"\n{'='*60}")
print(f"GPU: {name}")
print(f"Memory: {mem:.0f} GiB")
print(f"CUDA Version: {drv.group(1)}.{drv.group(2)}")
print(f"{'='*60}")

if mem < 22:
    print("❌ ERROR: Under 22 GiB. Cannot run Llama-3.1-8B.")
    print("   Switch to A100 runtime: Runtime -> Change runtime type -> A100")
elif mem < 38:
    print("⚠️  WARNING: Under 38 GiB. Batch size 8 may fail.")
    print("   Path B (batch 1) will work. For batch 8, switch to A100.")
else:
    print("✅ OK: A100 detected. Batch sizes 1, 8, 32 supported.")

In [ ]:
# ============================================================
# CELL 2: GOOGLE DRIVE + HF TOKEN
# ============================================================

import os
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/specdec"
os.makedirs(f"{DRIVE}/results", exist_ok=True)

# Get HF_TOKEN from Colab Secrets
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Secrets")
except userdata.SecretNotFoundError:
    raise SystemExit("❌ Add HF_TOKEN in Colab Secrets (🔑 icon in left sidebar)")
except userdata.NotebookAccessError:
    raise SystemExit("❌ Enable 'Notebook access' for HF_TOKEN in Secrets panel")

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["DRIVE"] = DRIVE
STORE = DRIVE
SAVE_MODELS = 1
SAVE_ENVS = 0
os.makedirs(STORE, exist_ok=True)
os.environ.update(STORE=STORE, SAVE_MODELS=str(SAVE_MODELS), SAVE_ENVS=str(SAVE_ENVS))

print(f"Results: {DRIVE}/results")
print(f"Store: {STORE}")

In [ ]:
%%bash
# ============================================================
# CELL 3: UNZIP CODE
# ============================================================
set -e
: "${DRIVE:?Run Cell 2 first}"
cd /content

UP="$DRIVE/specdec-bench.zip"
SAVED="$DRIVE/code/specdec-bench.zip"

if [ ! -d specdec-bench ]; then
    if [ -f "$SAVED" ]; then
        echo "Using saved code: $SAVED"
        unzip -q "$SAVED" -d /content
    elif [ -f "$UP" ]; then
        echo "Using uploaded code: $UP"
        unzip -q "$UP" -d /content
    else
        echo "❌ ERROR: Upload specdec-bench.zip to $UP"
        exit 1
    fi
fi

cd specdec-bench
rm -rf results 2>/dev/null || true
ln -sf "$DRIVE/results" results
echo "✅ Code ready"
ls -la

In [ ]:
%%bash
# ============================================================
# CELL 4: SETUP EAGLE ENVIRONMENT + DOWNLOAD MODELS
# ============================================================
# This takes ~5-10 minutes on first run
# ============================================================
set -e
cd /content/specdec-bench

# Restore saved models if available
python3 scripts/persist.py restore --store "$STORE" --what envs,models 2>/dev/null || true

# Setup Path B environment (EAGLE repo)
if [ ! -x .venv-eagle/bin/python ]; then
    echo "Setting up EAGLE environment (~5 min)..."
    bash envs/setup_eagle_env.sh > /content/setup_eagle.log 2>&1 || { tail -40 /content/setup_eagle.log; exit 1; }
fi
tail -2 /content/setup_eagle.log 2>/dev/null || echo "Environment ready"

# Run tests
echo ""
echo "Running tests..."
source .venv-eagle/bin/activate
EAGLE_REPO=third_party/EAGLE python -m unittest discover -s tests 2>&1 | tail -3
deactivate

# Download models
echo ""
echo "Downloading models (if not cached)..."
python3 -c "
from huggingface_hub import snapshot_download as s
for r in ['meta-llama/Llama-3.1-8B-Instruct', 'yuhuili/EAGLE3-LLaMA3.1-Instruct-8B', 'lmsys/sglang-EAGLE3-LLaMA3.1-Instruct-8B']:
    print(f'  Checking {r}...')
    s(r, ignore_patterns=['original/*','*.pth'])
"

# Save models to Drive
if [ "$SAVE_MODELS" = 1 ]; then
    python3 scripts/persist.py save --store "$STORE" --what models 2>/dev/null || echo "Models not saved (disk full?)"
fi

echo ""
echo "✅ Setup complete"
python3 scripts/persist.py status --store "$STORE"

---
## Path B: Batch Size 1 Only (EAGLE Repo)

Run Cells 5-9 if you want to test batch size 1 first.
**Skip to Cell 10 for batch size 8 testing.**

In [ ]:
%%bash
# ============================================================
# CELL 5: PATH B SMOKE TEST (batch 1)
# ============================================================
# Quick test: plain vs EAGLE-3 vs TALON (adaptive tree)
# ============================================================
cd /content/specdec-bench && source .venv-eagle/bin/activate
python scripts/quickstart_adaptive.py 2>&1 | grep -v -e GenerationMixin -e "^  - " -e "Loading checkpoint"

In [ ]:
%%bash
# ============================================================
# CELL 6: PATH B - 10 MT-Bench questions (batch 1)
# ============================================================
cd /content/specdec-bench && source .venv-eagle/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data

# Run EAGLE-3 + AR baseline
python scripts/run_eagle.py --data-dir $EAGLE_DATA --bench mt_bench --limit 10 --with-ar \
    --steps 9 --topk 10 --budget 60 --out-dir results/batch1_test --skip-existing \
    --eagle-repo third_party/EAGLE 2>&1 | grep -v -e GenerationMixin -e "^  - " -e "Loading checkpoint"

# Check losslessness
echo ""
python scripts/check_lossless.py results/batch1_test/mt_bench__s9_k10_b60__T0.jsonl results/batch1_test/mt_bench__ar__T0.jsonl

In [ ]:
%%bash
# ============================================================
# CELL 7: PATH B - TALON (adaptive tree) on same questions
# ============================================================
cd /content/specdec-bench && source .venv-eagle/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data

python scripts/run_eagle.py --data-dir $EAGLE_DATA --bench mt_bench --limit 10 \
    --adaptive --budget 60 --mu 0.03 --root-k 10 \
    --out-dir results/batch1_test --skip-existing \
    --eagle-repo third_party/EAGLE 2>&1 | grep -v -e GenerationMixin -e "^  - " -e "Loading checkpoint"

In [ ]:
# ============================================================
# CELL 8: COMPARE EAGLE-3 vs TALON (batch 1)
# ============================================================
import sys
sys.path.insert(0, "/content/specdec-bench")
from specdec.eagle_runner import read_jsonl
from specdec.metrics import throughput

D = "/content/specdec-bench/results/batch1_test/mt_bench__"
configs = {
    "Plain AR": "ar",
    "EAGLE-3 (static)": "s9_k10_b60",
    "TALON (adaptive)": "adaptive_n60_mu0.03_k10",
}

print(f"{'Method':<25} {'tok/s':>10} {'tau':>8} {'delta':>8} {'speedup':>10}")
print("=" * 65)

ar_tps = None
for name, tag in configs.items():
    try:
        _, rows = read_jsonl(f"{D}{tag}__T0.jsonl")
        tps = throughput(rows)
        if name == "Plain AR":
            ar_tps = tps
            print(f"{name:<25} {tps:>10.1f} {'-':>8} {'-':>8} {'-':>10}")
        else:
            tau = sum(r["new_tokens"] for r in rows) / sum(r["verify_steps"] for r in rows)
            delta = sum(r.get("draft_forwards", 0) for r in rows) / sum(r["verify_steps"] for r in rows)
            speedup = tps / ar_tps if ar_tps else 0
            print(f"{name:<25} {tps:>10.1f} {tau:>8.2f} {delta:>8.2f} {speedup:>9.2f}x")
    except Exception as e:
        print(f"{name:<25} (not run yet)")

print("\n" + "=" * 65)
print("tau = accepted tokens per verify pass (higher is better)")
print("delta = draft passes per verify pass (lower is better for TALON)")

In [ ]:
%%bash
# ============================================================
# CELL 9: PATH B FULL SUITE (many hours, batch 1 only)
# ============================================================
# Skip this if you want batch size 8 testing
# ============================================================
cd /content/specdec-bench && source .venv-eagle/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data
bash scripts/run_adaptive_suite.sh results/adaptive 2>&1 | tee -a results/adaptive_log.txt

---
## 🚀 Path A: Batch Size 8 Test (SGLang) - EAGLE-3 vs TALON

**THIS IS WHAT YOU WANT TO RUN!**

Tests **EAGLE-3 vs TALON-S** (adaptive tree) at batch size 8.

### What's Being Compared
| Method | Description | Parameters |
|--------|-------------|------------|
| EAGLE-3 (static) | Fixed tree shape, top-K selection | K=10, D=8, N=60 |
| TALON-S (adaptive) | Confidence-gated selection | N=60, mu=0.03, K=10 |

### TALON Algorithm 2
```
Layer 0: P_1 = Top-K children of root (K=10)
Layer d >= 1:
  S_d = all children of P_d
  m_d = max score in S_d (anchor)
  P_{d+1} = { u in S_d : p(u) >= mu * m_d }  (mu=0.03)
  Budget check: keep top (N - |T|) if needed
```

TALON-S is the SGLang variant with capped tree shapes (due to fixed tensor sizes).

### Run Order
1. Run Cell 10 to setup SGLang environment + register TALON plugin
2. Run Cell 11 to test batch size 8 with EAGLE-3 vs TALON
3. Run Cell 12 to view results
4. If successful, run Cell 13 for full sweep

In [ ]:
%%bash
# ============================================================
# CELL 10: SETUP SGLANG ENVIRONMENT + REGISTER TALON PLUGIN
# ============================================================
# Required for batch sizes > 1 and TALON-S
# Takes ~5-10 minutes on first run
# ============================================================
set -e
cd /content/specdec-bench

# Detect CUDA version and pick SGLang tag
DRV=$(nvidia-smi | grep -oP 'CUDA Version: \K[0-9]+')
if [ "$DRV" -ge 13 ]; then
    export SGLANG_TAG=v0.5.22
else
    export SGLANG_TAG=v0.5.19
fi
echo "CUDA $DRV -> SGLang $SGLANG_TAG"

# Setup SGLang environment
touch /content/setup_sglang.log
if [ ! -x .venv-sglang/bin/python ]; then
    echo "Setting up SGLang environment (~10 min)..."
    SGLANG_TAG=$SGLANG_TAG bash envs/setup_sglang_env.sh > /content/setup_sglang.log 2>&1 || {
        echo "Setup failed. Last 60 lines of log:"
        tail -60 /content/setup_sglang.log
        exit 1
    }
fi
tail -3 /content/setup_sglang.log 2>/dev/null || echo "SGLang ready"

# Ensure EAGLE data exists
[ -d third_party/EAGLE ] || git clone -q https://github.com/SafeAILab/EAGLE.git third_party/EAGLE

# ============================================================
# CRITICAL: Register the TALON plugin
# ============================================================
echo ""
echo "Registering TALON plugin..."
source .venv-sglang/bin/activate
pip install -e . -q

# Verify plugin registration
echo ""
echo "Verifying TALON plugin registration..."
python -c "
from importlib.metadata import entry_points
eps = list(entry_points(group='sglang.srt.plugins'))
names = [e.name for e in eps]
values = [e.value for e in eps]
if 'talon' in names:
    print('✅ TALON plugin registered:', values[names.index('talon')])
else:
    print('❌ TALON plugin NOT registered!')
    print('   Available plugins:', values)
    exit(1)
"

# Run TALON-specific tests
echo ""
echo "Running TALON tests..."
python -m unittest tests.test_talon_sglang tests.test_talon_sglang_plugin -v 2>&1 | tail -5

deactivate
echo ""
echo "✅ SGLang + TALON plugin ready for batch size 8 testing"

In [ ]:
%%bash
# ============================================================
# CELL 11: 🚀 BATCH SIZE 8 TEST - EAGLE-3 vs TALON
# ============================================================
# Compares:
#   1. EAGLE-3 (static): K=10, D=8, N=60 - top-K selection
#   2. TALON-S (adaptive): N=60, mu=0.03, K=10 - confidence-gated
#
# Uses batch_size=8, 20 questions each on mt_bench and humaneval
#
# TALON Algorithm 2:
#   Layer 0: Force top-K (K=10) children of root
#   Layer d>=1: Keep nodes where p(u) >= mu * m_d
#   Budget check at every layer
#
# Expected results:
#   - EAGLE-3: ~400-600 tok/s, tau ~5-7
#   - TALON-S: similar throughput, potentially better node selection
#
# Key insight from TALON paper:
#   - TALON creates deep-narrow trees for deterministic contexts
#   - TALON creates shallow-wide trees for uncertain contexts
#
# If this succeeds, proceed to Cell 13 for full sweep
# ============================================================
set -e
cd /content/specdec-bench
source .venv-sglang/bin/activate

export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data

echo "============================================================"
echo "BATCH SIZE 8 TEST: EAGLE-3 vs TALON"
echo "============================================================"
echo ""
echo "Comparison:"
echo "  ┌─────────────────────────────────────────────────────────┐"
echo "  │ EAGLE-3 (static)  : K=10, D=8, N=60, top-K selection    │"
echo "  │ TALON-S (adaptive): N=60, mu=0.03, K=10, conf-gated     │"
echo "  └─────────────────────────────────────────────────────────┘"
echo ""
echo "Benchmarks: mt_bench, humaneval"
echo "Questions: 20 per benchmark"
echo "Batch size: 8"
echo "============================================================"
echo ""

# Run the test with EAGLE-3 vs TALON-S
python scripts/run_sglang_sweep.py \
    --config configs/batch8_test.yaml \
    --data-dir $EAGLE_DATA \
    --out-dir results/batch8_test \
    --skip-existing

echo ""
echo "============================================================"
echo "✅ BATCH SIZE 8 TEST COMPLETE"
echo "============================================================"
echo "Run Cell 12 to view results"
echo "If successful, run Cell 13 for full sweep (batch 1/8/32)"

In [ ]:
# ============================================================
# CELL 12: VIEW BATCH SIZE 8 RESULTS - EAGLE-3 vs TALON
# ============================================================
import json
import os

results_dir = "/content/specdec-bench/results/batch8_test"

print("=" * 80)
print("BATCH SIZE 8 RESULTS: EAGLE-3 vs TALON-S")
print("=" * 80)
print()
print("TALON Algorithm 2 (Liu et al., arXiv 2601.07353):")
print("  Layer 0: Force top-K (K=10) children of root")
print("  Layer d>=1: Keep nodes where p(u) >= mu * m_d (mu=0.03)")
print()

# Collect results by benchmark and method
results = {}
for server_dir in sorted(os.listdir(results_dir)):
    server_path = os.path.join(results_dir, server_dir)
    if not os.path.isdir(server_path):
        continue

    for jsonl_file in sorted(os.listdir(server_path)):
        if not jsonl_file.endswith(".jsonl"):
            continue

        filepath = os.path.join(server_path, jsonl_file)
        try:
            with open(filepath) as f:
                lines = f.readlines()

            if len(lines) >= 2:
                meta = json.loads(lines[0])
                summary_line = json.loads(lines[-1])
                summary = summary_line.get("_summary", summary_line)

                bench = meta.get("bench", "unknown")

                # Determine method from server name
                if "talon" in server_dir.lower():
                    method = "TALON-S"
                elif "eagle3" in server_dir.lower():
                    method = "EAGLE-3"
                else:
                    method = server_dir

                results[(bench, method)] = {
                    "throughput": summary.get("throughput_full_occupancy", summary.get("throughput_all", 0)),
                    "tau": summary.get("tau", "-"),
                    "errors": summary.get("n_errors", 0),
                    "server": server_dir,
                }
        except Exception as e:
            pass

# Print comparison table
for bench in ["mt_bench", "humaneval"]:
    print(f"\n{'─'*80}")
    print(f"  {bench.upper()}")
    print(f"{'─'*80}")
    print(f"{'Method':<20} {'Throughput':>15} {'tau':>10} {'Errors':>10} {'Server Dir'}")
    print(f"{'─'*80}")

    eagle_tps = None
    for method in ["EAGLE-3", "TALON-S"]:
        if (bench, method) in results:
            r = results[(bench, method)]
            tau_str = f"{r['tau']:.2f}" if isinstance(r['tau'], (int, float)) else str(r['tau'])
            tps = r['throughput']

            if method == "EAGLE-3":
                eagle_tps = tps
                speedup_str = ""
            elif eagle_tps and eagle_tps > 0:
                speedup = tps / eagle_tps
                speedup_str = f" ({speedup:.2f}x vs EAGLE-3)"
            else:
                speedup_str = ""

            print(f"{method:<20} {tps:>12.1f}/s {tau_str:>10} {r['errors']:>10}  {r['server']}{speedup_str}")
        else:
            print(f"{method:<20} (not run yet)")

print("\n" + "=" * 80)
print("Key metrics:")
print("  throughput : tokens per second (higher is better)")
print("  tau        : accepted tokens per verify pass (higher is better)")
print("  errors     : should be 0")
print()
print("TALON insight:")
print("  - Confidence-gated selection adapts tree shape to context difficulty")
print("  - Deep-narrow trees for deterministic contexts (e.g., code completion)")
print("  - Shallow-wide trees for uncertain contexts (e.g., creative writing)")
print()
print("If results look good, proceed to Cell 13 for full sweep.")

In [ ]:
%%bash
# ============================================================
# CELL 13: FULL SWEEP - EAGLE-3 vs TALON (all batch sizes)
# ============================================================
# Only run this AFTER Cell 11 succeeds!
#
# This runs:
#   - 6 benchmarks: mt_bench, humaneval, gsm8k, alpaca, sum, qa
#   - 3 batch sizes: 1, 8, 32
#   - Multiple configurations:
#     * AR baseline
#     * SGLang auto
#     * Static EAGLE-3 (5 tree configs)
#     * TALON-S at mu=0.03 (5 matching trees)
#     * TALON-S at mu=0.01, 0.1 on (9,10,60)
#
# Takes several hours. Resumable if interrupted.
# ============================================================
set -e
cd /content/specdec-bench
source .venv-sglang/bin/activate

export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data

echo "============================================================"
echo "FULL SWEEP: EAGLE-3 vs TALON"
echo "============================================================"
echo ""
echo "Configurations:"
echo "  - AR baseline (speedup denominator)"
echo "  - SGLang auto-tuned"
echo "  - Static EAGLE-3: 5 tree configs"
echo "  - TALON-S mu=0.03: 5 matching trees"
echo "  - TALON-S mu=0.01, 0.1: on (9, 10, 60)"
echo ""
echo "Benchmarks: mt_bench, humaneval, gsm8k, alpaca, sum, qa"
echo "Batch sizes: 1, 8, 32"
echo ""
echo "This will take several hours. Progress is saved."
echo "Re-run this cell to resume if interrupted."
echo "============================================================"
echo ""

python scripts/run_sglang_sweep.py \
    --config configs/sweep_talon_sglang.yaml \
    --data-dir $EAGLE_DATA \
    --out-dir results/talon_sglang \
    --skip-existing 2>&1 | tee -a results/talon_sglang_log.txt

---
## Results & Summary

In [ ]:
%%bash
# ============================================================
# CELL 14: GENERATE SUMMARY CSV
# ============================================================
cd /content/specdec-bench
source .venv-eagle/bin/activate
python scripts/summarize.py results/ --out results/summary.csv
echo ""
echo "Summary saved to: results/summary.csv"

In [ ]:
# ============================================================
# CELL 15: DISPLAY ALL RESULTS
# ============================================================
import pandas as pd

try:
    df = pd.read_csv(f"{DRIVE}/results/summary.csv")
    pd.set_option("display.max_rows", 200, "display.max_columns", 40)
    display(df)
except FileNotFoundError:
    print("No summary.csv yet. Run Cell 14 first.")

In [ ]:
# ============================================================
# CELL 16: EAGLE-3 vs TALON-S COMPARISON TABLE
# ============================================================
import pandas as pd
import os

try:
    df = pd.read_csv(f"{DRIVE}/results/summary.csv")

    # Filter for EAGLE-3 and TALON results
    eagle_mask = df['config'].str.contains('eagle3', case=False, na=False)
    talon_mask = df['config'].str.contains('talon', case=False, na=False)

    comparison = df[eagle_mask | talon_mask][[
        'bench', 'batch_size', 'config', 'throughput_tok_s', 'tau', 'speedup'
    ]].sort_values(['bench', 'batch_size', 'config'])

    print("EAGLE-3 vs TALON-S Comparison")
    print("=" * 80)
    display(comparison)

except FileNotFoundError:
    print("No summary.csv yet. Run Cell 14 first.")
except Exception as e:
    print(f"Error loading results: {e}")

In [ ]:
%%bash
# ============================================================
# CELL 17: SAVE CODE TO DRIVE
# ============================================================
cd /content/specdec-bench
WHAT=code
[ "$SAVE_ENVS" = 1 ] && WHAT=code,envs
python3 scripts/persist.py save --store "$STORE" --what $WHAT
python3 scripts/persist.py status --store "$STORE"
echo ""
echo "✅ Code saved to Drive"